In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('/kaggle/input/datasets/ahmedmohameddawoud/ecommerce-ab-testing/ab_test.csv')
df.head()

,id,time,con_treat,page,converted
0,851104,11:48.6,control,old_page,0
1,804228,01:45.2,control,old_page,0
2,661590,55:06.2,treatment,new_page,0
3,853541,28:03.1,treatment,new_page,0
4,864975,52:26.2,control,old_page,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         294478 non-null  int64 
 1   time       294478 non-null  object
 2   con_treat  294478 non-null  object
 3   page       294478 non-null  object
 4   converted  294478 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 11.2+ MB


In [4]:
df.isna().sum()

id           0
time         0
con_treat    0
page         0
converted    0
dtype: int64

In [5]:
df['id'].nunique()

290584

In [6]:
len(df)

294478

In [7]:
duplicates = df[df.duplicated('id', keep=False)]

duplicates.sort_values('id').head(20)

,id,time,con_treat,page,converted
213114,630052,25:54.1,treatment,old_page,1
230259,630052,16:05.2,treatment,new_page,0
251762,630126,16:00.3,treatment,new_page,0
22513,630126,35:54.8,treatment,old_page,0
11792,630137,59:22.1,control,new_page,0
183371,630137,08:49.9,control,old_page,0
255753,630320,27:37.2,treatment,old_page,0
207211,630320,02:43.6,control,old_page,0
110634,630471,42:51.5,control,old_page,0
96929,630471,14:17.4,control,new_page,0


In [8]:
group_per_user = (
    df.groupby('id')['con_treat']
      .nunique()
)

bad_users = group_per_user[group_per_user > 1]

print(f'Количество пользователей в обеих группах: {len(bad_users)}')

Количество пользователей в обеих группах: 1895


In [9]:
bad_pct = len(bad_users) / df['id'].nunique()

print(f'Доля ошибок: {bad_pct:.2%}')

Доля ошибок: 0.65%


In [10]:
invalid_rows = df[
    ((df['con_treat'] == 'control') &
     (df['page'] == 'new_page'))
    |
    ((df['con_treat'] == 'treatment') &
     (df['page'] == 'old_page'))
]

invalid_rows.shape

(3893, 5)

In [11]:
df.shape

(294478, 5)

In [12]:
df['id'].nunique()

290584

In [13]:
len(bad_users)

1895

In [14]:
invalid_rows.shape

(3893, 5)

In [15]:
bad_ids = set(bad_users.index)

invalid_ids = set(invalid_rows['id'])

len(bad_ids.intersection(invalid_ids))

1895

In [16]:
df_clean = df[
    ~df['id'].isin(bad_users.index)
].copy()

In [17]:
df_clean.groupby('id')['con_treat'].nunique().max()

1

In [18]:
df_clean.shape

(290688, 5)

**В ходе аудита качества данных были обнаружены 1 895 пользователей (0.65%), одновременно присутствующие в контрольной и тестовой группах. Также выявлено 3 893 наблюдения с несоответствием между назначенной группой и отображаемой версией страницы. Такие наблюдения были удалены из анализа во избежание contamination bias и нарушения предпосылок A/B-тестирования.**

In [19]:
from scipy.stats import chisquare

group_counts = df_clean['con_treat'].value_counts()

group_counts

con_treat
treatment    145381
control      145307
Name: count, dtype: int64

In [20]:
observed = group_counts.values

expected = [len(df_clean)/2,
            len(df_clean)/2]

chi2, p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

print("Chi2 =", chi2)
print("p-value =", p_value)

Chi2 = 0.01883806693086746
p-value = 0.8908317380724055


**Sample Ratio Mismatch (SRM) не обнаружен. Распределение пользователей между контрольной и тестовой группами соответствует ожидаемому соотношению 50/50 (χ²=0.019, p=0.891). Следовательно, отсутствуют признаки систематической ошибки в механизме рандомизации эксперимента.**

In [21]:
df_clean['time'].head()

0    11:48.6
1    01:45.2
2    55:06.2
3    28:03.1
4    52:26.2
Name: time, dtype: object

In [22]:
df_clean['time'].sample(10)

93557     02:45.5
52244     18:36.6
219400    30:35.3
39999     55:16.7
198815    53:00.6
207675    26:49.3
226002    39:13.5
83656     19:28.8
159659    04:55.0
278259    36:54.8
Name: time, dtype: object

**Исходный датасет не содержит календарной даты проведения эксперимента, поэтому анализ длительности теста и влияния day-of-week effect выполнить невозможно. Ограничение зафиксировано в разделе Data Limitations.**

In [23]:
ab_summary = (
    df_clean
    .groupby('con_treat')
    .agg(
        users=('id','count'),
        conversions=('converted','sum'),
        cr=('converted','mean')
    )
)

ab_summary

,users,conversions,cr
con_treat,,,
control,145307,17487,0.120345
treatment,145381,17290,0.118929


In [24]:
ab_summary = (
    df_clean
    .groupby('con_treat')
    .agg(
        users=('id', 'count'),
        conversions=('converted', 'sum')
    )
)

ab_summary['cr'] = (
    ab_summary['conversions']
    / ab_summary['users']
)

ab_summary

,users,conversions,cr
con_treat,,,
control,145307,17487,0.120345
treatment,145381,17290,0.118929


In [25]:
cr_control = ab_summary.loc['control', 'cr']
cr_treatment = ab_summary.loc['treatment', 'cr']

absolute_uplift = cr_treatment - cr_control

relative_uplift = (
    absolute_uplift
    / cr_control
)

print(f'CR Control: {cr_control:.4%}')
print(f'CR Treatment: {cr_treatment:.4%}')

print(f'Absolute uplift: {absolute_uplift:.4%}')
print(f'Relative uplift: {relative_uplift:.2%}')

CR Control: 12.0345%
CR Treatment: 11.8929%
Absolute uplift: -0.1416%
Relative uplift: -1.18%


In [26]:
from statsmodels.stats.proportion import proportions_ztest

successes = [
    ab_summary.loc['control', 'conversions'],
    ab_summary.loc['treatment', 'conversions']
]

samples = [
    ab_summary.loc['control', 'users'],
    ab_summary.loc['treatment', 'users']
]

z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=samples
)

print("Z-stat =", z_stat)
print("p-value =", p_value)

Z-stat = 1.176469036591924
p-value = 0.23940749849829834


In [27]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(
    df_clean['con_treat'],
    df_clean['converted']
)

contingency

converted,0,1
con_treat,,
control,127820,17487
treatment,128091,17290


In [28]:
chi2, p_chi, dof, expected = chi2_contingency(
    contingency
)

print("Chi2 =", chi2)
print("p-value =", p_chi)

Chi2 = 1.3706648047719256
p-value = 0.24169769389763893


In [29]:
from statsmodels.stats.proportion import confint_proportions_2indep

low, high = confint_proportions_2indep(
    count1=ab_summary.loc['treatment', 'conversions'],
    nobs1=ab_summary.loc['treatment', 'users'],
    count2=ab_summary.loc['control', 'conversions'],
    nobs2=ab_summary.loc['control', 'users'],
    method='wald'
)

print(f'95% CI: ({low:.5f}, {high:.5f})')

95% CI: (-0.00378, 0.00094)



**Статистический анализ показал отсутствие значимых различий между контрольной и тестовой версиями посадочной страницы. Конверсия контрольной версии составила 12.03%, тогда как новая версия показала конверсию 11.89%.Относительное изменение конверсии составило -1.18%. Проверка гипотез с использованием Z-теста для пропорций (p-value = 0.239) и критерия хи-квадрат (p-value = 0.242) не выявила статистически значимого эффекта.95%-й доверительный интервал для разницы конверсий находится в диапазоне от -0.378 процентных пункта до +0.094 процентных пункта и включает нулевое значение, что дополнительно подтверждает отсутствие доказанного улучшения.На основании проведенного анализа отсутствуют статистические основания для раскатывания нового дизайна на 100% пользователей.**


In [30]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p1 = cr_control
p2 = cr_treatment

effect_size = proportion_effectsize(
    p1,
    p2
)

power_analysis = NormalIndPower()

power = power_analysis.power(
    effect_size=abs(effect_size),
    nobs1=ab_summary.loc['control', 'users'],
    ratio=1,
    alpha=0.05
)

print("Effect size:", effect_size)
print("Power:", power)

Effect size: 0.004364137661615919
Power: 0.21748100542014207


In [31]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from scipy.optimize import brentq

baseline_cr = cr_control

analysis = NormalIndPower()

n = ab_summary.loc['control', 'users']

def solve_mde(delta):
    effect = proportion_effectsize(
        baseline_cr,
        baseline_cr + delta
    )

    return (
        analysis.power(
            effect_size=effect,
            nobs1=n,
            ratio=1,
            alpha=0.05
        ) - 0.8
    )

mde = brentq(
    solve_mde,
    0.0001,
    0.05
)

print(f"MDE = {mde:.4%}")

MDE = 0.3402%


**Новый дизайн не показал статистически значимого улучшения конверсии. Наблюдаемое изменение составляет −1.18% относительно контрольной версии. Доверительный интервал включает как небольшой отрицательный, так и небольшой положительный эффект, поэтому решение о полном раскатывании нового интерфейса на основании данного эксперимента принимать не рекомендуется.**

**Статистически значимых различий между вариантами не обнаружено. Наблюдаемый эффект оказался меньше минимально детектируемого эффекта (MDE), поэтому эксперимент не дает достаточных оснований для раскатки нового дизайна.**

In [32]:
np.random.seed(42)

df_unit = df_clean.copy()

df_unit['revenue'] = 0.0

In [33]:
control_mask = (
    (df_unit['con_treat'] == 'control')
    &
    (df_unit['converted'] == 1)
)

df_unit.loc[control_mask, 'revenue'] = np.random.lognormal(
    mean=np.log(250000),
    sigma=0.7,
    size=control_mask.sum()
)

In [34]:
treatment_mask = (
    (df_unit['con_treat'] == 'treatment')
    &
    (df_unit['converted'] == 1)
)

df_unit.loc[treatment_mask, 'revenue'] = np.random.lognormal(
    mean=np.log(255000),
    sigma=0.75,
    size=treatment_mask.sum()
)

In [35]:
df_unit.groupby('con_treat')['revenue'].describe()

,count,mean,std,min,25%,50%,75%,max
con_treat,,,,,,,,
control,145307.0,38602.905587,137518.186749,0.0,0.0,0.0,0.0,5.749222e+06
treatment,145381.0,40047.075846,148559.822078,0.0,0.0,0.0,0.0,4.905007e+06


In [36]:
MARGIN = 0.40

CAC = 15000

In [37]:
df_unit['gross_profit'] = (
    df_unit['revenue']
    * MARGIN
)

In [38]:
df_unit['acquisition_cost'] = CAC

In [39]:
df_unit['profit'] = (
    df_unit['gross_profit']
    - df_unit['acquisition_cost']
)

In [40]:
df_unit.groupby('con_treat').agg(
    users=('id','count'),
    revenue=('revenue','sum'),
    profit=('profit','sum')
)

,users,revenue,profit
con_treat,,,
control,145307,5.609272e+09,6.410396e+07
treatment,145381,5.822084e+09,1.481186e+08


In [41]:
summary = (
    df_unit
    .groupby('con_treat')
    .agg(
        users=('id','count'),
        payers=('converted','sum'),
        revenue=('revenue','sum'),
        profit=('profit','sum')
    )
)

summary['ARPU'] = (
    summary['revenue']
    / summary['users']
)

summary['ARPPU'] = (
    summary['revenue']
    / summary['payers']
)

summary['ROI'] = (
    summary['profit']
    /
    (summary['users'] * CAC)
)

summary['ROMI'] = (
    (summary['profit'])
    /
    (summary['users'] * CAC)
)

summary

,users,payers,revenue,profit,ARPU,ARPPU,ROI,ROMI
con_treat,,,,,,,,
control,145307,17487,5.609272e+09,6.410396e+07,38602.905587,320768.136451,0.029411,0.029411
treatment,145381,17290,5.822084e+09,1.481186e+08,40047.075846,336731.285918,0.067922,0.067922


In [42]:
summary['Profit_per_User'] = (
    summary['profit']
    / summary['users']
)

summary

,users,payers,revenue,profit,ARPU,ARPPU,ROI,ROMI,Profit_per_User
con_treat,,,,,,,,,
control,145307,17487,5.609272e+09,6.410396e+07,38602.905587,320768.136451,0.029411,0.029411,441.162235
treatment,145381,17290,5.822084e+09,1.481186e+08,40047.075846,336731.285918,0.067922,0.067922,1018.830338


In [43]:
future_users = 1_000_000

In [44]:
control_profit_per_user = (
    summary.loc['control','profit']
    /
    summary.loc['control','users']
)

treatment_profit_per_user = (
    summary.loc['treatment','profit']
    /
    summary.loc['treatment','users']
)

In [45]:
future_control_profit = (
    control_profit_per_user
    * future_users
)

future_treatment_profit = (
    treatment_profit_per_user
    * future_users
)

incremental_profit = (
    future_treatment_profit
    - future_control_profit
)

print(future_control_profit)
print(future_treatment_profit)
print(incremental_profit)

441162234.7500768
1018830338.2889128
577668103.538836


In [46]:
summary

,users,payers,revenue,profit,ARPU,ARPPU,ROI,ROMI,Profit_per_User
con_treat,,,,,,,,,
control,145307,17487,5.609272e+09,6.410396e+07,38602.905587,320768.136451,0.029411,0.029411,441.162235
treatment,145381,17290,5.822084e+09,1.481186e+08,40047.075846,336731.285918,0.067922,0.067922,1018.830338


In [47]:
future_users = 1_000_000

control_ppu = summary.loc['control', 'Profit_per_User']
treatment_ppu = summary.loc['treatment', 'Profit_per_User']

future_control_profit = control_ppu * future_users
future_treatment_profit = treatment_ppu * future_users

incremental_profit = (
    future_treatment_profit
    - future_control_profit
)

print(f"Control: {future_control_profit:,.0f}")
print(f"Treatment: {future_treatment_profit:,.0f}")
print(f"Incremental: {incremental_profit:,.0f}")

Control: 441,162,235
Treatment: 1,018,830,338
Incremental: 577,668,104


In [48]:
np.random.seed(42)
start_date = '2026-06-01'
end_date = '2026-06-14'

date_range = pd.date_range(start=start_date, end=end_date).strftime('%Y-%m-%d').tolist()

df_unit['date'] = np.random.choice(date_range, size=len(df_unit))

df_unit['date'] = pd.to_datetime(df_unit['date'])

print(df_unit['date'].value_counts().sort_index())

date
2026-06-01    20799
2026-06-02    20791
2026-06-03    20594
2026-06-04    20661
2026-06-05    20861
2026-06-06    20817
2026-06-07    20872
2026-06-08    20574
2026-06-09    20614
2026-06-10    21060
2026-06-11    20549
2026-06-12    20897
2026-06-13    20882
2026-06-14    20717
Name: count, dtype: int64


In [49]:
df_unit.to_csv('ecommerce_ab_unit_economics.csv', index=False)
print("Файл успешно сохранен в Outputs ноутбука!")

Файл успешно сохранен в Outputs ноутбука!


In [50]:
df_unit

,id,time,con_treat,page,converted,revenue,gross_profit,acquisition_cost,profit,date
0,851104,11:48.6,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-07
1,804228,01:45.2,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-04
2,661590,55:06.2,treatment,new_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-13
3,853541,28:03.1,treatment,new_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-11
4,864975,52:26.2,control,old_page,1,353951.828068,141580.731227,15000,126580.731227,2026-06-08
...,...,...,...,...,...,...,...,...,...,...
294473,751197,28:38.6,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-04
294474,945152,51:57.1,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-01
294475,734608,45:03.4,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-04
294476,697314,20:29.0,control,old_page,0,0.000000,0.000000,15000,-15000.000000,2026-06-06
